# Navier–Stokes: conditioning of the Gram matrix

We learn the map from a forcing $f$ to the vorticity $w(\cdot, T)$ of the 2-D incompressible Navier–Stokes equations on the torus (vorticity form, $w(0) = 0$, $\mathrm{Re} = 500$, $T = 5$), from a fixed dataset of 10000 training and 2000 test pairs on a $128 \times 128$ grid ([Zenodo 12825163](https://zenodo.org/records/12825163); see `raw_data/README.md`). We cannot choose where the inputs are, so the input measure is the empirical measure of the training forcings and we use `owls.EmpiricalWLS`, which samples optimally from the dataset.

* Inputs: PCA coefficients of the forcing that keep 99.5% of the variance (`encode_inputs`), scaled to $[-1, 1]$. Each coefficient gets a symmetric Jacobi measure with matching variance; it only selects the polynomial family (the basis is orthonormalized on the data).
* Outputs: Fourier coefficients in a basis orthonormal in $H^\alpha$, keeping 99.9% of the average $H^\alpha$ energy (`encode_outputs`).
* Approximation space: polynomial operators with an anisotropic hyperbolic cross index set, weights $w_j = \sigma_j / \sigma_1$ (normalized PCA singular values), univariate degree at most 10.

For hyperbolic cross sets of increasing size we draw $M = \lfloor N\log N\rfloor$ training pairs either from the discrete optimal measure (leverage scores) or uniformly from the dataset, and record $\mathrm{cond}(\mathbf G)$ (3 repetitions, $L^2$ outputs). The figure omits the largest set ($k = 24$).

**Status:** the data in `data/` are from the arXiv v1 run; this experiment was not rerun after the cleanup. Rerunning needs the dataset.

In [ ]:
RECOMPUTE = False  # True: rerun the experiment and overwrite data/ (see ../README.md for run times)
QUICK = False      # True: tiny version of the experiment, written to a temporary folder (for testing)

import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import owls
from owls.plotting import paper_style, savefig
from ns_utils import load_ns_data, encode_inputs, encode_outputs

OUT = tempfile.mkdtemp() if QUICK else "."
DATA, FIGS = os.path.join(OUT, "data"), os.path.join(OUT, "figures")
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

In [ ]:
# Parameters (as run)
HC = np.array([6, 9, 12, 15, 18, 21, 24])
RUNS = 3
ALPHA = 0            # L^2 outputs
MAX_DEGREE = 10
N_MAX = 5000
SEED = 0
if QUICK:
    HC, RUNS = np.array([3, 4, 5]), 2

In [ ]:
if RECOMPUTE or QUICK:
    X_train, Y_train, X_test, Y_test = load_ns_data("raw_data")
    if QUICK:
        X_train, Y_train, X_test, Y_test = X_train[:600], Y_train[:600], X_test[:100], Y_test[:100]
    x_train, x_test, pca = encode_inputs(X_train, X_test, energy=0.995, random_state=SEED)
    measures = owls.alpha_from_variance(np.var(x_train, axis=0))   # symmetric Jacobi parameters matching each variance
    weights = pca.singular_values_ / pca.singular_values_[0]        # anisotropy weights
    print(f"d_in = {x_train.shape[1]} PCA modes")
    y_train, y_test, compressor = encode_outputs(Y_train, Y_test, alpha=ALPHA, energy=0.999)

    dims = np.zeros(len(HC))
    cond = {True: np.zeros((len(HC), RUNS)), False: np.zeros((len(HC), RUNS))}
    for i, k in enumerate(HC):
        L = owls.index_set(x_train.shape[1], k, "hc", weights, MAX_DEGREE, N_MAX)
        N = len(L)
        dims[i] = N
        model = owls.EmpiricalWLS(x_train, y_train, L, measures)   # the QR factorization is done once here
        for r in range(RUNS):
            for optimal in (True, False):
                model.fit(int(N * np.log(N)), optimal=optimal, rng=np.random.default_rng([SEED, i, r, int(optimal)]))
                cond[optimal][i, r] = model.cond()
        print(f"k = {k}: N_eff = {N}, mean cond(G) optimal {cond[True][i].mean():.2f}, uniform {cond[False][i].mean():.2e}")
    np.save(os.path.join(DATA, "Condition_Ind.npy"), cond[True])
    np.save(os.path.join(DATA, "Condition_Std.npy"), cond[False])
    np.save(os.path.join(DATA, "Condition_Dims.npy"), dims)

In [ ]:
C_ind = np.load(os.path.join(DATA, "Condition_Ind.npy"))[:-1]   # the figure omits the largest set
C_std = np.load(os.path.join(DATA, "Condition_Std.npy"))[:-1]
dims = np.load(os.path.join(DATA, "Condition_Dims.npy"))[:-1]
hc = HC[:-1]

fig_width, fig_height = paper_style(width=0.5, aspect=0.6, fontsize=7, legend_fontsize=5)
fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout="constrained")
ax.set_yscale("log")
for vals, style, label in [(C_std, "kv--", r"Standard Sampling: $f^i\sim \tau$"), (C_ind, "r^--", r"Induced Sampling: $f^i\sim \mu$")]:
    for i, N in enumerate(dims):
        ax.plot([N, N], [vals[i].min(), vals[i].max()], "k--", linewidth=0.5)
        ax.plot(np.full(vals.shape[1], N), vals[i], "ko", markersize=1)
    ax.plot(dims, vals.mean(axis=1), style, markersize=3, linewidth=1, label=label)
ax.set_xlabel("\n" + r"$N_{\mathrm{eff}}(k)$ for $ k\in \{$" + f"{hc[0]}, {hc[1]}, ..., {hc[-1]}" + r"$\}$")
ax.set_ylabel(r"$\mathrm{cond}(\mathbf{G})$")
ax.legend(loc="best")
savefig(os.path.join(FIGS, "NS_CondNum.pdf"))
plt.show()